# 04 - Sensitivity Analysis

**Purpose:** Analyze how cost savings vary with battery capacity and other parameters.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.config.settings import load_config
from src.data.loaders import generate_synthetic_data
from src.evaluation.sensitivity import run_sensitivity_analysis
from src.plots.visualizations import set_plot_style, plot_sensitivity_results

%matplotlib inline
set_plot_style()

## Load Data

In [ ]:
settings = load_config('../config/config.yaml')

df = generate_synthetic_data(
    start_date=settings.start_date,
    end_date=settings.end_date,
    resolution_minutes=settings.resolution_minutes,
    solar_capacity_kw=settings.solar_capacity_kw,
    timezone=settings.timezone,
)
print(f"Data loaded: {len(df)} time steps")

## Sensitivity: Battery Capacity

In [ ]:
capacity_values = [0, 5, 10, 13.5, 20, 30, 50]

capacity_results = run_sensitivity_analysis(
    base_settings=settings,
    parameter_name='battery.capacity_kwh',
    parameter_values=capacity_values,
    timestamps=df.index,
    prices=df['price'].values,
    p_solar=df['solar'].values,
    p_load=df['load'].values,
)

print("Battery Capacity Sensitivity:")
print(capacity_results.to_dataframe().to_string(index=False))

In [ ]:
fig = plot_sensitivity_results(capacity_results)
plt.suptitle('Battery Capacity Sensitivity Analysis', y=1.02, fontsize=14)
plt.show()

## Sensitivity: Battery Power Rating

In [ ]:
power_values = [1, 2, 3, 5, 7, 10]

power_results = run_sensitivity_analysis(
    base_settings=settings,
    parameter_name='battery.max_power_kw',
    parameter_values=power_values,
    timestamps=df.index,
    prices=df['price'].values,
    p_solar=df['solar'].values,
    p_load=df['load'].values,
)

fig = plot_sensitivity_results(power_results)
plt.suptitle('Battery Power Rating Sensitivity Analysis', y=1.02, fontsize=14)
plt.show()

## Sensitivity: Feed-in Tariff

In [ ]:
fit_values = [0, 0.03, 0.05, 0.08, 0.10, 0.15]

fit_results = run_sensitivity_analysis(
    base_settings=settings,
    parameter_name='grid.feed_in_tariff',
    parameter_values=fit_values,
    timestamps=df.index,
    prices=df['price'].values,
    p_solar=df['solar'].values,
    p_load=df['load'].values,
)

fig = plot_sensitivity_results(fit_results)
plt.suptitle('Feed-in Tariff Sensitivity Analysis', y=1.02, fontsize=14)
plt.show()

## Key Findings

1. **Diminishing returns** - Larger batteries show diminishing savings beyond ~20 kWh
2. **Power rating matters** - Insufficient power limits arbitrage opportunity
3. **FiT impact** - Lower feed-in tariffs increase battery value (more to gain from self-consumption)

In [ ]:
# Summary table
summary = pd.DataFrame({
    'Parameter': ['Battery Capacity', 'Battery Power', 'Feed-in Tariff'],
    'Optimal Value': [
        f"{capacity_values[np.argmax(capacity_results.savings_pct)]} kWh",
        f"{power_values[np.argmax(power_results.savings_pct)]} kW",
        f"${fit_values[np.argmax(fit_results.savings_pct)]}/kWh",
    ],
    'Max Savings (%)': [
        f"{max(capacity_results.savings_pct):.1f}%",
        f"{max(power_results.savings_pct):.1f}%",
        f"{max(fit_results.savings_pct):.1f}%",
    ],
})

print("\nSensitivity Analysis Summary:")
print(summary.to_string(index=False))